# Z-Score Normalized PSTH Heatmap Analysis

This notebook creates Z-score normalized heatmaps that combine PSTH data from multiple units.
- **Y-axis**: Units (sorted in ascending order)
- **X-axis**: Time relative to interval start (ms)
- **Color**: Z-score normalized firing rate (standard deviations from unit mean)

## Z-Score Normalization Formula
**z = (x - u) / o**
- **z** = z-score
- **x** = one bin's value (frequency)
- **u** = mean of all bins for the same unit
- **o** = standard deviation from mean of all bins from the same unit

This normalization allows for better comparison across units with different baseline firing rates.

## Parameters
Configure the analysis parameters in the cell below and run the analysis.

In [ ]:
# Import required modules
import sys
import os
sys.path.append('.')

# Force reload of the module to pick up new changes
import importlib
if 'normalized' in sys.modules:
    importlib.reload(sys.modules['normalized'])

import normalized
from normalized import (
    create_multiple_duration_normalized_heatmaps,
    analyze_units_by_max_zscore
)
import matplotlib.pyplot as plt


In [ ]:
# Configuration Parameters

# Data files (adjust paths as needed)
spikes_file = '../../Data/040425/spikes.csv'
intervals_file = '../../Data/040425/pico_time_adjust.csv'
spikes_path = os.path.abspath(spikes_file)
intervals_path = os.path.abspath(intervals_file)

# Analysis parameters
durations_ms = [5, 10, 25]        # Interval durations to analyze (ms)
units = None                      # Units to include (None = all units)
bin_size_ms = 0.1               # Bin size in milliseconds
pre_interval_ms = 5               # Time before interval start (ms)
post_interval_ms = 10             # Time after interval END (ms)
smooth_window = 10                 # Smoothing window (bins)

# Whisking exclusion parameters
exclude_whisking = True          # Set to True to exclude intervals overlapping with whisking
whisking_file = '../../Data/040425/whisking_video_time.csv'  # Path to whisking.csv file (if it exists)
# Note: whisking.csv should contain 'Start' and 'End' columns with interval times to exclude

# Output settings
save_plots = True
output_dir = '../../Output/040425/normalized_heatmaps_exclude_whisking'

In [ ]:
# Create Z-Score Normalized PSTH heatmaps for all specified durations

save_dir = output_dir if save_plots else None

results = create_multiple_duration_normalized_heatmaps(
    spikes_file=spikes_path,
    intervals_file=intervals_path,
    durations_ms=durations_ms,
    units=units,
    bin_size_ms=bin_size_ms,
    pre_interval_ms=pre_interval_ms,
    post_interval_ms=post_interval_ms,
    smooth_window=smooth_window,
    exclude_whisking=exclude_whisking,
    whisking_file=whisking_file if exclude_whisking else None,
    save_dir=save_dir
)

# Show the plots
plt.show()

In [ ]:
# Analyze units by their highest maximum z-score values
unit_rankings = analyze_units_by_max_zscore(results, display_results=True)

In [ ]:
# Test whisking exclusion functionality
print("=== Testing Whisking Exclusion Functionality ===")

# Check if the 040425 dataset has whisking data
test_whisking_file = '../../Data/040425/whisking_video_time.csv'
if os.path.exists(test_whisking_file):
    print(f"Found whisking file: {test_whisking_file}")
    
    # Test with the 040425 dataset
    test_spikes_file = '../../Data/040425/spikes.csv'
    test_intervals_file = '../../Data/040425/pico_time_adjust.csv'
    
    if os.path.exists(test_spikes_file) and os.path.exists(test_intervals_file):
        print("Running test with whisking exclusion enabled...")
        
        # Test with a small subset for quick validation
        test_results = create_multiple_duration_normalized_heatmaps(
            spikes_file=os.path.abspath(test_spikes_file),
            intervals_file=os.path.abspath(test_intervals_file),
            durations_ms=[5],  # Just test one duration
            units=[1, 2, 3],   # Just test a few units
            bin_size_ms=0.6,
            pre_interval_ms=5,
            post_interval_ms=10,
            smooth_window=5,
            exclude_whisking=True,
            whisking_file=test_whisking_file,
            save_dir=None  # Don't save during test
        )
        
        if test_results and 5 in test_results and test_results[5][0] is not None:
            print("✓ Whisking exclusion test passed!")
            print(f"Generated heatmap with whisking intervals excluded")
        else:
            print("✗ Whisking exclusion test failed - no results generated")
    else:
        print("Missing required files for 040425 dataset")
else:
    print(f"Whisking file not found at {test_whisking_file}")
    print("Whisking exclusion functionality cannot be tested with available data")